# CLIP + RAG + DistilBERT (GossipCop)

Fusion de `CLIP_papier.ipynb` (DistilBERT + ResNet-50 + similarite CLIP + MLP) et de `RAG_BERT.ipynb` (DuckDuckGo, cache, reranking MiniLM, texte `NEWS` + preuves web pour DistilBERT).

- **DistilBERT** : entre sur `text_for_bert` (`model_input` RAG ou baseline sans preuves).
- **CLIP** (texte court) : **300 premiers caracteres du `content` seul** (pas des preuves web), pour garder une similarite image–news.
- Memes contraintes que le notebook CLIP : pas de troncature DistilBERT — les textes trop longs en tokens sont exclus.

In [ ]:
!pip install lime

In [ ]:
import warnings
warnings.filterwarnings("ignore")

!pip install open_clip_torch
!pip install ddgs
!pip install sentence-transformers

In [ ]:
import json
import os
import random
import re
import time
import nltk
import open_clip
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Any

from PIL import Image
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel

from ddgs import DDGS
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

nltk.download("stopwords", quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

## Configuration

Adapter `BASE_DIR`, chemins CSV et dossiers `gossip_train` / `gossip_test`.

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/PFE/AAAI_dataset")
IMAGE_DIR = Path("/content/drive/MyDrive/PFE/AAAI_dataset/Images")
TRAIN_CSV = BASE_DIR / "gossip_train.csv"

TEXT_COL = "content"
IMAGE_COL = "image"
LABEL_COL = "label"
TEXT_FOR_BERT_COL = "text_for_bert"

TEXT_MODEL_NAME = "distilbert-base-uncased"
CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"

MAX_LENGTH = 512
IMAGE_SIZE = 224
BATCH_SIZE = 12
NUM_EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

CACHE_DIR = BASE_DIR / "clip_rag_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RERANK_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_WEB_RESULTS = 8
TOP_EVIDENCES = 3
SLEEP_BETWEEN_SEARCHES_SEC = 1.0
MAX_QUERY_CHARS = 200
MAX_SNIPPET_CHARS = 100
MAX_NEWS_CHARS = 900
MIN_HIT_TITLE_CHARS = 12

USE_RAG = True

TRAIN_BALANCE = 200
VAL_BALANCE = 50
TEST_BALANCE = 50

EXPERIMENT_TAG = (
    f"clip_rag_mw={MAX_WEB_RESULTS}_te={TOP_EVIDENCES}"
    f"_qc={MAX_QUERY_CHARS}_sc={MAX_SNIPPET_CHARS}_nc={MAX_NEWS_CHARS}"
    f"_tr={TRAIN_BALANCE}_val={VAL_BALANCE}_ts={TEST_BALANCE}")

print("CACHE_DIR =", CACHE_DIR)
print("EXPERIMENT_TAG =", EXPERIMENT_TAG)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

def clean_text(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"Click to share on [^.]+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Opens in new window", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

STOPWORDS = set(stopwords.words("english"))

def path_image(df: pd.DataFrame, name_folder: str) -> pd.DataFrame:
    df = df.copy()
    folder = Path(IMAGE_DIR) / name_folder
    df[IMAGE_COL] = df[IMAGE_COL].apply(lambda x: folder / str(x))
    df = df[df[IMAGE_COL].apply(lambda p: Path(p).exists())].copy()
    df[IMAGE_COL] = df[IMAGE_COL].astype(str)
    return df

def text_is_valid(text: str, max_length: int = MAX_LENGTH) -> bool:
    text = clean_text(text)
    tokens = tokenizer(text, truncation=False, add_special_tokens=True)
    return len(tokens["input_ids"]) <= max_length

def filter_df(df: pd.DataFrame, image_folder: str) -> pd.DataFrame:
    df = df[[TEXT_COL, IMAGE_COL, LABEL_COL]].dropna().copy()
    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)
    df = path_image(df, image_folder)
    df = df[df[TEXT_COL].apply(text_is_valid)].copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df.reset_index(drop=True)

def balance_df(df: pd.DataFrame, target_size: int) -> pd.DataFrame:
    target_size = int(target_size)
    per_class = target_size // 2
    fake_df = df[df[LABEL_COL] == 1]
    real_df = df[df[LABEL_COL] == 0]
    n = min(per_class, len(fake_df), len(real_df))
    balanced = pd.concat([
            fake_df.sample(n=n, random_state=SEED),
            real_df.sample(n=n, random_state=SEED),
        ]
    ).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return balanced

train_df = filter_df(pd.read_csv(TRAIN_CSV), "gossip_train")
train_df["label"] = train_df["label"].map({0: 1, 1: 0})

train_df, test_df = train_test_split(train_df, test_size=0.5, random_state=42, stratify=train_df[LABEL_COL])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df[LABEL_COL])

train_df = balance_df(train_df, TRAIN_BALANCE)
test_df = balance_df(test_df, TEST_BALANCE)
val_df = balance_df(val_df, VAL_BALANCE)

print("Train :", train_df.shape)
print("Val   :", val_df.shape)
print("Test  :", test_df.shape)
print("Train labels:", train_df[LABEL_COL].value_counts().to_dict())
print("Val labels  :", val_df[LABEL_COL].value_counts().to_dict())
print("Test labels :", test_df[LABEL_COL].value_counts().to_dict())
train_df.head()

## RAG : requete, DuckDuckGo, cache, reranking

In [ ]:
def _unique_keep_order(items: list[str]) -> list[str]:
    out = []
    seen = set()
    for item in items:
        k = item.strip().lower()
        if not k or k in seen:
            continue
        seen.add(k)
        out.append(item.strip())
    return out


def extract_candidates(text: str) -> list[str]:
    text = clean_text(text)
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    first_sentence = clean_text(sentences[0]) if sentences else ""
    first_words = " ".join(text.split()[:18])
    capitalized = re.findall(r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})\b", text)
    capitalized = [c.strip() for c in capitalized if len(c.strip()) > 3]
    words = re.findall(r"[A-Za-z][A-Za-z\-']+", text.lower())
    keywords = []
    seen = set()
    for w in words:
        if len(w) < 4 or w in STOPWORDS or w in seen:
            continue
        seen.add(w)
        keywords.append(w)
        if len(keywords) == 5:
            break
    query_parts = _unique_keep_order([
            first_sentence[:110],
            first_words[:90],
            " ".join(capitalized[:3]),
            " ".join(keywords[:5])])
    query = " ".join(part for part in query_parts if part)
    query = clean_text(query)
    query = query[:MAX_QUERY_CHARS].rsplit(" ", 1)[0]
    return [query] if query else []


def search_web_ddg(query: str, max_results: int = MAX_WEB_RESULTS) -> list[dict[str, str]]:
    results: list[dict[str, str]] = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, region="wt-wt", safesearch="off", max_results=max_results):
                title = clean_text(r.get("title", ""))
                snippet = clean_text(r.get("body", ""))[:MAX_SNIPPET_CHARS]
                if len(title) < MIN_HIT_TITLE_CHARS or not snippet:
                    continue
                results.append({
                        "title": title,
                        "snippet": snippet,
                        "url": clean_text(r.get("href", ""))})
    except Exception as e:
        print(f"[DDG] {query[:80]} -> {e}")
    return results

def cache_path(split_name: str, expected_size: int) -> Path:
    return CACHE_DIR / f"retrieval_{split_name}_{expected_size}_{EXPERIMENT_TAG}.jsonl"

def is_compatible_cache(rows: list[dict[str, Any]], df: pd.DataFrame) -> bool:
    if len(rows) != len(df):
        return False
    if not rows:
        return True
    sample_idx = min(2, len(rows) - 1)
    return clean_text(rows[sample_idx].get("content", "")) == clean_text(df.iloc[sample_idx][TEXT_COL])

def run_retrieval(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    cache_file = cache_path(split_name, len(df))
    rows: list[dict[str, Any]] = []

    if cache_file.exists():
        with cache_file.open("r", encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        if is_compatible_cache(rows, df):
            print(f"Cache charge: {cache_file.name} ({len(rows)} lignes)")
            return pd.DataFrame(rows)
        print(f"Cache ignore (incompatible): {cache_file.name}")

    for idx in tqdm(range(len(df)), desc=f"Retrieval {split_name}"):
        content = clean_text(df.at[idx, TEXT_COL])
        query_candidates = extract_candidates(content)
        query = query_candidates[0] if query_candidates else " ".join(content.split()[:20])
        hits = search_web_ddg(query)
        rows.append(
            {
                "content": content,
                "label": int(df.at[idx, LABEL_COL]),
                "query": query,
                "n_hits": len(hits),
                "hits": hits})
        time.sleep(SLEEP_BETWEEN_SEARCHES_SEC)

    with cache_file.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Cache ecrit: {cache_file}")
    return pd.DataFrame(rows)

In [ ]:
reranker = SentenceTransformer(RERANK_MODEL, device=str(DEVICE))

def rerank_hits(
    content: str, hits: list[dict[str, str]], top_k: int = TOP_EVIDENCES
) -> list[dict[str, Any]]:
    if not hits:
        return []
    news_text = clean_text(content)[:MAX_NEWS_CHARS]
    doc_texts = [f"{h.get('title', '')}. {h.get('snippet', '')}".strip() for h in hits]
    news_emb = reranker.encode([news_text], normalize_embeddings=True)
    doc_embs = reranker.encode(doc_texts, normalize_embeddings=True)
    scores = (doc_embs @ news_emb[0]).tolist()
    ranked = []
    for hit, score in zip(hits, scores):
        ranked.append({**hit, "score": float(score)})
    ranked.sort(key=lambda x: x["score"], reverse=True)
    return ranked[:top_k]

def build_model_input(
    content: str, evidences: list[dict[str, Any]], use_rag: bool = True
) -> str:
    news = clean_text(content)[:MAX_NEWS_CHARS]
    pieces = [
        "TASK: classify the celebrity news as real (0) or fake (1).",
        f"NEWS: {news}",
    ]
    if use_rag:
        for i, ev in enumerate(evidences, start=1):
            title = clean_text(ev.get("title", ""))
            snippet = clean_text(ev.get("snippet", ""))
            if title or snippet:
                pieces.append(f"WEB_EVIDENCE_{i}: {title}. {snippet}")
    return " [SEP] ".join(pieces)


def add_evidence_columns(df_ret: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df_ret.iterrows(), total=len(df_ret), desc="Reranking"):
        use_rag = USE_RAG
        evidences = (
            rerank_hits(row["content"], row["hits"], top_k=TOP_EVIDENCES)
            if use_rag
            else []
        )
        rows.append(
            {
                "content": row["content"],
                "label": int(row["label"]),
                "query": row["query"],
                "n_hits": int(row["n_hits"]),
                "top_evidences": evidences,
                "model_input": build_model_input(row["content"], evidences, use_rag=use_rag),
            }
        )
    return pd.DataFrame(rows)

def attach_rag_text(mm_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Ajoute `text_for_bert` aligne sur les lignes du dataframe multimodal."""
    sub = mm_df[[TEXT_COL, LABEL_COL]].copy()
    if USE_RAG:
        ret = run_retrieval(sub, split_name)
        ready = add_evidence_columns(ret)
    else:
        rows = []
        for _, r in sub.iterrows():
            c = clean_text(r[TEXT_COL])
            rows.append(
                {
                    "content": c,
                    "label": int(r[LABEL_COL]),
                    "model_input": build_model_input(c, [], use_rag=False),
                }
            )
        ready = pd.DataFrame(rows)

    merged = mm_df.merge(ready[["content", "model_input"]], on=TEXT_COL, how="inner")
    merged = merged.rename(columns={"model_input": TEXT_FOR_BERT_COL})
    merged = merged[merged[TEXT_FOR_BERT_COL].apply(text_is_valid)].reset_index(drop=True)
    return merged

train_df = attach_rag_text(train_df, "train")
val_df = attach_rag_text(val_df, "val")
test_df = attach_rag_text(test_df, "test")

print("Apres RAG + filtre tokens — Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())

## Dataset multimodal, CLIP, modele, entrainement

In [ ]:
CLIP_CHARS = 300

image_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model = clip_model.to(DEVICE)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

class GossipCopDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        bert_text = row[TEXT_FOR_BERT_COL]
        clip_text_src = row[TEXT_COL]
        image = Image.open(row[IMAGE_COL]).convert("RGB")
        label = int(row[LABEL_COL])

        text_inputs = tokenizer(
            bert_text,
            padding="max_length",
            truncation=False,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        clip_text_input = clip_text_src[:CLIP_CHARS]
        clip_text = clip_tokenizer([clip_text_input])

        return {
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "image": image_transform(image),
            "clip_image": clip_preprocess(image),
            "clip_text": clip_text.squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }


def safe_collate(batch):
    keys = batch[0].keys()
    out = {}
    for k in keys:
        vals = [b[k] for b in batch]
        if torch.is_tensor(vals[0]):
            out[k] = torch.stack([v.contiguous() for v in vals], dim=0)
        else:
            out[k] = vals
    return out


train_dataset = GossipCopDataset(train_df)
val_dataset = GossipCopDataset(val_df)
test_dataset = GossipCopDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=safe_collate,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=safe_collate,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=safe_collate,
)

In [ ]:
class SimpleFNDModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME)
        text_hidden = self.text_encoder.config.hidden_size

        self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        image_hidden = self.image_encoder.fc.in_features
        self.image_encoder.fc = nn.Identity()

        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for _, param in self.image_encoder.named_parameters():
            param.requires_grad = False
        for param in self.text_encoder.transformer.layer[-2:].parameters():
            param.requires_grad = True
        for param in self.image_encoder.layer4.parameters():
            param.requires_grad = True

        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.image_proj = nn.Sequential(
            nn.Linear(image_hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 + 256 + 1, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2),
        )

    def forward(self, input_ids, attention_mask, image, clip_image, clip_text):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feature = text_outputs.last_hidden_state[:, 0, :]
        text_feature = self.text_proj(text_feature)

        image_feature = self.image_encoder(image)
        image_feature = self.image_proj(image_feature)

        with torch.no_grad():
            clip_image_features = clip_model.encode_image(clip_image)
            clip_text_features = clip_model.encode_text(clip_text)
            clip_image_features = clip_image_features / clip_image_features.norm(dim=-1, keepdim=True)
            clip_text_features = clip_text_features / clip_text_features.norm(dim=-1, keepdim=True)
            clip_similarity = (clip_image_features * clip_text_features).sum(dim=1, keepdim=True)

        fused = torch.cat([text_feature, image_feature, clip_similarity], dim=1)
        return self.classifier(fused)


model = SimpleFNDModel().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY
)
print(model.__class__.__name__)

In [ ]:
def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    all_labels = []
    all_preds = []
    total_loss = 0.0

    for batch in loader:
        batch = move_batch_to_device(batch, DEVICE)
        with torch.set_grad_enabled(training):
            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                image=batch["image"],
                clip_image=batch["clip_image"],
                clip_text=batch["clip_text"],
            )
            loss = criterion(logits, batch["label"])
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_loss += loss.item() * batch["label"].size(0)
        all_labels.extend(batch["label"].detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


best_val_acc = -1.0
best_model_path = BASE_DIR / "best_clip_rag_bert.pt"

for epoch in range(NUM_EPOCHS):
    train_metrics = run_epoch(model, train_loader, optimizer)
    val_metrics = run_epoch(model, val_loader)
    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_metrics[0]:.4f} train_acc={train_metrics[1]:.4f} | "
        f"val_loss={val_metrics[0]:.4f} val_acc={val_metrics[1]:.4f}"
    )
    if val_metrics[1] > best_val_acc:
        best_val_acc = val_metrics[1]
        torch.save(model.state_dict(), best_model_path)

print("Meilleure accuracy validation :", best_val_acc)

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_acc = run_epoch(model, test_loader)
print("Test accuracy  :", round(test_acc, 4))

model.eval()
all_y = []
all_p = []
with torch.no_grad():
    for batch in test_loader:
        batch = move_batch_to_device(batch, DEVICE)
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            image=batch["image"],
            clip_image=batch["clip_image"],
            clip_text=batch["clip_text"],
        )
        all_p.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_y.extend(batch["label"].cpu().numpy())

print(classification_report(all_y, all_p, digits=4))
print("F1 macro:", f1_score(all_y, all_p, average="macro"))
print("Accuracy :", accuracy_score(all_y, all_p))
print(confusion_matrix(all_y, all_p))